In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
from scipy.stats import linregress

In [2]:
fund = pd.read_csv("../data/processed/01_fund_master_cleaned.csv")

nav = pd.read_csv("../data/processed/02_nav_history_cleaned.csv")

performance = pd.read_csv("../data/processed/07_scheme_performance_cleaned.csv")

benchmark = pd.read_csv("../data/processed/10_benchmark_indices_cleaned.csv")

In [3]:
nav["date"] = pd.to_datetime(nav["date"])

benchmark["date"] = pd.to_datetime(benchmark["date"])

In [4]:
print("Fund Master:", fund.shape)

print("NAV:", nav.shape)

print("Performance:", performance.shape)

print("Benchmark:", benchmark.shape)

Fund Master: (40, 15)
NAV: (46000, 3)
Performance: (40, 19)
Benchmark: (8050, 3)


In [5]:
nav = nav.sort_values(
    ["amfi_code", "date"]
)

nav["daily_return"] = (
    nav.groupby("amfi_code")["nav"]
       .pct_change()
)

In [6]:
nav.head(10)

,amfi_code,date,nav,daily_return
0,100016,2022-01-03,520.4608,NaN
1,100016,2022-01-04,515.0971,-0.010306
2,100016,2022-01-05,521.7239,0.012865
3,100016,2022-01-06,515.7880,-0.011377
4,100016,2022-01-07,515.1639,-0.001210
5,100016,2022-01-10,510.7136,-0.008639
6,100016,2022-01-11,513.5542,0.005562
7,100016,2022-01-12,512.3195,-0.002404
8,100016,2022-01-13,510.2445,-0.004050
9,100016,2022-01-14,514.3636,0.008073


In [7]:
# Get first and last NAV for each fund
cagr = nav.groupby("amfi_code").agg(
    start_nav=("nav", "first"),
    end_nav=("nav", "last")
).reset_index()

# Calculate CAGR
cagr["CAGR_1yr"] = (cagr["end_nav"] / cagr["start_nav"]) ** (1/1) - 1
cagr["CAGR_3yr"] = (cagr["end_nav"] / cagr["start_nav"]) ** (1/3) - 1
cagr["CAGR_5yr"] = (cagr["end_nav"] / cagr["start_nav"]) ** (1/5) - 1

cagr.head()

,amfi_code,start_nav,end_nav,CAGR_1yr,CAGR_3yr,CAGR_5yr
0,100016,520.4608,583.6113,0.121336,0.038912,0.023168
1,100025,26.3169,31.8843,0.211552,0.066058,0.039127
2,100033,107.3758,342.0072,2.185142,0.471328,0.260741
3,101206,305.0996,773.2939,1.534562,0.363435,0.204427
4,101207,38.5736,53.9836,0.399496,0.118555,0.069533


In [8]:
cagr.to_csv("../reports/cagr_table.csv", index=False)

print("✅ CAGR table saved")

✅ CAGR table saved


In [9]:
risk_free_rate = 0.065

sharpe = nav.groupby("amfi_code")["daily_return"].agg(["mean", "std"]).reset_index()

sharpe["Sharpe_Ratio"] = (
    (sharpe["mean"] * 252 - risk_free_rate)
    / (sharpe["std"] * np.sqrt(252))
)

sharpe.head()

,amfi_code,mean,std,Sharpe_Ratio
0,100016,0.000142,0.009164,-0.201517
1,100025,0.000170,0.002460,-0.567095
2,100033,0.001080,0.011929,1.093699
3,101206,0.000852,0.009177,1.027213
4,101207,0.000424,0.016251,0.162661


In [10]:
sortino_list = []

for code in nav["amfi_code"].unique():
    data = nav[nav["amfi_code"] == code]

    downside = data[data["daily_return"] < 0]["daily_return"]

    downside_std = downside.std()

    mean_return = data["daily_return"].mean()

    ratio = (mean_return * 252 - risk_free_rate) / (downside_std * np.sqrt(252))

    sortino_list.append([code, ratio])

sortino = pd.DataFrame(sortino_list, columns=["amfi_code", "Sortino_Ratio"])

sortino.head()

,amfi_code,Sortino_Ratio
0,100016,-0.351047
1,100025,-0.941821
2,100033,1.829134
3,101206,1.799563
4,101207,0.276644


In [11]:
# Benchmark daily returns
benchmark = benchmark.sort_values("date")
benchmark["benchmark_return"] = benchmark.groupby("index_name")["close_value"].pct_change()

# Use Nifty 100 benchmark
nifty100 = benchmark[benchmark["index_name"].str.contains("100", case=False, na=False)]

alpha_beta = []

for code in nav["amfi_code"].unique():

    fund = nav[nav["amfi_code"] == code]

    merged = pd.merge(
        fund,
        nifty100[["date", "benchmark_return"]],
        on="date",
        how="inner"
    )

    merged = merged.dropna()

    if len(merged) > 30:

        slope, intercept, r, p, std = linregress(
            merged["benchmark_return"],
            merged["daily_return"]
        )

        alpha = intercept * 252
        beta = slope

    else:
        alpha = np.nan
        beta = np.nan

    alpha_beta.append([code, alpha, beta])

alpha_beta = pd.DataFrame(
    alpha_beta,
    columns=["amfi_code", "Alpha", "Beta"]
)

alpha_beta.head()

,amfi_code,Alpha,Beta
0,100016,0.037476,-0.058268
1,100025,0.042818,0.001158
2,100033,0.271954,0.005104
3,101206,0.213998,0.021086
4,101207,0.108971,-0.065289


In [12]:
alpha_beta.to_csv("../reports/alpha_beta.csv", index=False)

print("✅ Alpha Beta CSV Saved")

✅ Alpha Beta CSV Saved


In [13]:
drawdown = []

for code in nav["amfi_code"].unique():

    df = nav[nav["amfi_code"] == code].copy()

    df["running_max"] = df["nav"].cummax()

    df["drawdown"] = df["nav"] / df["running_max"] - 1

    drawdown.append([
        code,
        df["drawdown"].min()
    ])

drawdown = pd.DataFrame(
    drawdown,
    columns=["amfi_code", "Max_Drawdown"]
)

drawdown.head()

,amfi_code,Max_Drawdown
0,100016,-0.247344
1,100025,-0.043083
2,100033,-0.162172
3,101206,-0.112916
4,101207,-0.354469


In [14]:
score = performance.merge(
    sharpe[["amfi_code", "Sharpe_Ratio"]],
    on="amfi_code"
)

score = score.merge(
    alpha_beta,
    on="amfi_code"
)

score = score.merge(
    drawdown,
    on="amfi_code"
)

score["Score"] = (
    score["return_3yr_pct"].rank(pct=True) * 30
    + score["Sharpe_Ratio"].rank(pct=True) * 25
    + score["Alpha"].rank(pct=True) * 20
    + (1 / score["expense_ratio_pct"]).rank(pct=True) * 15
    + (1 / abs(score["Max_Drawdown"])).rank(pct=True) * 10
)

score.head()

,amfi_code,scheme_name,fund_house,category,plan,return_1yr_pct,return_3yr_pct,return_5yr_pct,benchmark_3yr_pct,alpha,...,max_drawdown_pct,aum_crore,expense_ratio_pct,morningstar_rating,risk_grade,Sharpe_Ratio,Alpha,Beta,Max_Drawdown,Score
0,119551,SBI Bluechip Fund - Regular Plan - Growth,SBI Mutual Fund,Large Cap,Regular,12.42,12.36,14.45,11.49,0.87,...,-21.70,14288,1.54,4,Moderate,1.208267,0.232010,-0.031751,-0.150124,59.8125
1,119552,SBI Bluechip Fund - Direct Plan - Growth,SBI Mutual Fund,Large Cap,Direct,15.25,11.30,14.23,9.52,1.78,...,-24.43,1231,0.66,3,Moderate,0.953279,0.198686,-0.026159,-0.118035,57.7500
2,119598,SBI Small Cap Fund - Regular Plan - Growth,SBI Mutual Fund,Small Cap,Regular,24.56,23.39,20.67,22.16,1.23,...,-13.35,19259,1.43,5,Very High,0.945308,0.303370,-0.023196,-0.287060,75.6250
3,119599,SBI Small Cap Fund - Direct Plan - Growth,SBI Mutual Fund,Small Cap,Direct,20.59,23.14,21.82,22.01,1.13,...,-24.78,36061,0.72,4,Very High,-0.057187,0.048824,0.062002,-0.525742,50.9375
4,119120,SBI Magnum Gilt Fund - Regular Plan - Growth,SBI Mutual Fund,Gilt,Regular,5.34,6.07,5.43,4.47,1.60,...,-2.30,24101,0.77,5,Low,-0.226575,0.056209,-0.006414,-0.043287,29.2500


In [15]:
score.to_csv("../reports/fund_scorecard.csv", index=False)

print("✅ Fund Scorecard Saved")

✅ Fund Scorecard Saved


In [16]:
top5 = score.nlargest(5, "Score")["amfi_code"]

plot_data = nav[nav["amfi_code"].isin(top5)]

fig = px.line(
    plot_data,
    x="date",
    y="nav",
    color="amfi_code",
    title="Top 5 Funds vs Benchmark"
)

fig.show()

fig.write_image("../reports/charts/benchmark_comparison.png")

print("✅ Benchmark Chart Saved")

✅ Benchmark Chart Saved
